# Notebook to run nerfstudio experiments
- Add new experiments by appending to `EXPERIMENTS`
- Add any new CLI flag by putting it in `extra_args` (gets mapped to `--flag value`)
- Add nested method args via `method_args`
- Set `DRY_RUN = True` to preview commands without executing

# Imports

In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime

import os
import subprocess
import sys
import time
import itertools
import shutil
import yaml
from itertools import product

# Workspace Setup

In [ ]:
# Check conda environment
env_name = os.environ.get("CONDA_DEFAULT_ENV", "unknown")
print(f"Conda environment: {env_name}")
print(f"Python: {sys.executable}")
print(f"ns-train found: {shutil.which('ns-train')}")

# For cleaner logs because nerfstudio uses rich for terminal output
os.environ["NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"

# Paths
WORKSPACE_DIR = "/home/islabella/workspaces/irwin_ws/fyp-playground"
DATASETS = {
    "torpedo_unprocessed": os.path.join(WORKSPACE_DIR, "datasets", "torpedo", "torpedo_unprocessed"),
    "saltpond_unprocessed": os.path.join(WORKSPACE_DIR, "datasets", "saltpond", "saltpond_unprocessed"),
}
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")
LOG_DIR = os.path.join(WORKSPACE_DIR, "logs")

os.chdir(WORKSPACE_DIR)
print(f"Working directory: {os.getcwd()}")

# Experiments Configuration

In [ ]:
# Experiment templates - model and params only (independent of dataset)
EXPERIMENT_TEMPLATES = [
    {
        "suffix": "repeat_baseline",
        "extra_args": {},
    },
    # {
    #     "suffix": "no-seathru",
    #     "extra_args": {
    #         "pipeline.model.do-seathru": False,
    #         "pipeline.model.learn-background": False,
    #         "pipeline.model.background-color": "random",
    #         "pipeline.model.output-depth-during-training": False,
    #         "pipeline.model.add-recon-depth-l1": False,
    #         "pipeline.model.use-depth-smooth-loss": False,
    #         "pipeline.model.use-alpha-smooth-loss": False,
    #         "pipeline.model.use-opacity-prior": False,
    #         "pipeline.model.use-dcp-loss": False,
    #         "pipeline.model.use-rgb-sat-loss": False,
    #         "pipeline.model.use-gw-loss": False,
    #         "pipeline.model.use-rgb-sv-loss": False,
    #         "pipeline.model.use-binf-loss": False,
    #         "pipeline.model.use-dsc-attenuation-loss": False,
    #         "pipeline.model.use-depth-l1-loss": False,
    #         "pipeline.model.use-depth-weighted-l1": False,
    #         "pipeline.model.use-depth-weighted-l2": False,
    #     },
    # },
    # {
    #     "suffix": "no-seathru",
    #     "extra_args": {
    #         "pipeline.model.do-seathru": False,
    #         "pipeline.model.learn-background": False,
    #         "pipeline.model.background-color": "random",
    #         "pipeline.model.output-depth-during-training": False,
    #         "pipeline.model.add-recon-depth-l1": False,
    #         "pipeline.model.use-depth-smooth-loss": False,
    #         "pipeline.model.use-alpha-smooth-loss": False,
    #         "pipeline.model.use-opacity-prior": False,
    #         "pipeline.model.use-dcp-loss": False,
    #         "pipeline.model.use-rgb-sat-loss": False,
    #         "pipeline.model.use-gw-loss": False,
    #         "pipeline.model.use-rgb-sv-loss": False,
    #         "pipeline.model.use-binf-loss": False,
    #         "pipeline.model.use-dsc-attenuation-loss": False,
    #         "pipeline.model.use-depth-l1-loss": False,
    #         "pipeline.model.use-depth-weighted-l1": False,
    #         "pipeline.model.use-depth-weighted-l2": False,
    #         "pipeline.model.cull-alpha-thresh": 0.005,
    #         "pipeline.model.use-scale-regularization": False,
    #     },
    # },
    # {
    #     "suffix": "no-seathru",
    #     "extra_args": {
    #         "pipeline.model.do-seathru": False,
    #         "pipeline.model.learn-background": False,
    #         "pipeline.model.background-color": "random",
    #         "pipeline.model.output-depth-during-training": False,
    #         "pipeline.model.add-recon-depth-l1": False,
    #         "pipeline.model.use-depth-smooth-loss": False,
    #         "pipeline.model.use-alpha-smooth-loss": False,
    #         "pipeline.model.use-opacity-prior": False,
    #         "pipeline.model.use-dcp-loss": False,
    #         "pipeline.model.use-rgb-sat-loss": False,
    #         "pipeline.model.use-gw-loss": False,
    #         "pipeline.model.use-rgb-sv-loss": False,
    #         "pipeline.model.use-binf-loss": False,
    #         "pipeline.model.use-dsc-attenuation-loss": False,
    #         "pipeline.model.use-depth-l1-loss": False,
    #         "pipeline.model.use-depth-weighted-l1": False,
    #         "pipeline.model.use-depth-weighted-l2": False,
    #         "pipeline.model.cull-alpha-thresh": 0.005,
    #         "pipeline.model.use-scale-regularization": True,
    #     },
    # },
]

# * Grid searches
# MODELS = ["splatfacto"]
MODELS = ["sea-splatfacto"]
# MODELS = ["splatfacto", "sea-splatfacto"]

EXPERIMENTS = []
number_of_repeats = 3
for dataset_name, dataset_path in DATASETS.items():
    print(f"{dataset_path} exists: {Path(dataset_path).exists()}")
    for template in EXPERIMENT_TEMPLATES:
        for _ in range(number_of_repeats):
            name = f"{dataset_name}/{template['suffix']}"
            for model in MODELS:
                EXPERIMENTS.append({
                    "name": name,
                    "model": model,
                    "data": dataset_path,
                    "output_dir": os.path.join(OUTPUT_DIR, dataset_name),
                    "vis": "viewer+tensorboard",
                    "viewer": False,
                    "extra_args": {
                        "experiment-name": f"{dataset_name}-{template['suffix']}",
                        **template["extra_args"],
                    },
            })

DRY_RUN = False  # start with True to preview commands first

## To run subsets

In [ ]:
# Only one dataset
# EXPERIMENTS = [e for e in EXPERIMENTS if "torpedo_unprocessed" in e["name"]]

# Only baselines across all datasets
# EXPERIMENTS = [e for e in EXPERIMENTS if e["name"].endswith("no-seathru")]

# Specific combo
# EXPERIMENTS = [e for e in EXPERIMENTS if e["name"] == "red-sea-wreck/no-priors"]

# Multiple specific templates
# KEEP = {"baseline", "no-seathru", "backscatter-only"}
# EXPERIMENTS = [e for e in EXPERIMENTS if e["name"].split("/")[1] in KEEP]

print(f"Running {len(EXPERIMENTS)} experiments")

# Command builder

In [ ]:
def build_command(experiment: dict) -> list[str]:
    """Build an ns-train CLI command from an experiment config dict.

    Any key in extra_args automatically becomes a --flag value pair.
    """
    cmd = ["ns-train", experiment["model"]]

    cmd += ["--data", str(experiment["data"])]

    output_dir = experiment.get("output_dir", "./outputs")
    cmd += ["--output-dir", str(output_dir)]

    vis = experiment.get("vis", "viewer")
    cmd += ["--vis", vis]
    
    if experiment.get("viewer") is False:
        cmd += ["--viewer.quit-on-train-completion", "True"]

    for key, value in experiment.get("extra_args", {}).items():
        cmd += [f"--{key}", str(value)]

    for key, value in experiment.get("method_args", {}).items():
        cmd += [f"--{key}", str(value)]

    return cmd


def command_to_str(cmd: list[str]) -> str:
    """Pretty-print a command list as a shell string."""
    return " ".join(cmd)

## Preview commands

In [ ]:
for i, exp in enumerate(EXPERIMENTS):
    cmd = build_command(exp)
    print(f"[{i}] {exp['name']}")
    print(f"    {command_to_str(cmd)}")
    print()

# Run experiments

In [ ]:
results = []
LOG_ALL_EXPERIMENTS = True

for i, exp in enumerate(EXPERIMENTS):
    cmd = build_command(exp)
    name = exp["name"]
    print(f"[{i+1}/{len(EXPERIMENTS)}] Starting: {name}")
    
    if DRY_RUN:
        results.append({"name": name, "status": "skipped (dry run)", "duration": 0})
        continue
    
    Path(exp.get("output_dir", "./outputs")).mkdir(parents=True, exist_ok=True)
    dataset_name = name.split("/")[0]
    log_subdir = Path(LOG_DIR) / dataset_name
    log_subdir.mkdir(parents=True, exist_ok=True)
    log_file = log_subdir / f"{exp['extra_args']['experiment-name']}_{i}.log" if LOG_ALL_EXPERIMENTS else log_subdir / f"{exp['extra_args']['experiment-name']}.log"
    
    start = time.time()
    try:
        with open(log_file, "w") as f:
            proc = subprocess.run(
                cmd,
                check=True,
                text=True,
                stdout=f,
                stderr=subprocess.STDOUT,
            )
        status = "success"
    except subprocess.CalledProcessError as e:
        status = f"failed (exit code {e.returncode})"
        # Save failed log with a unique name
        failed_log_file = log_subdir / f"{i}_{exp['extra_args']['experiment-name']}_FAILED_{int(time.time())}.log"
        shutil.copy2(log_file, failed_log_file)
        log_file = failed_log_file  # Update log_file reference for results
    except FileNotFoundError:
        status = "failed (ns-train not found)"
        failed_log_file = log_subdir / f"{i}_{exp['extra_args']['experiment-name']}_FAILED_{int(time.time())}.log"
        if log_file.exists():
            shutil.copy2(log_file, failed_log_file)
            log_file = failed_log_file
    
    duration = time.time() - start
    results.append({"name": name, "status": status, "duration": duration, "log": str(log_file)})
    print(f"  → {status} ({duration/60:.1f} min) — log: {log_file}")

# Training Summary

In [ ]:
print(f"\n{'Experiment':<40} {'Status':<25} {'Duration':>10}")
print("-" * 75)
for r in results:
    dur = f"{r['duration']/60:.1f} min" if r['duration'] > 0 else "—"
    print(f"{r['name']:<40} {r['status']:<25} {dur:>10}")

# Reading Experiment Configs

In [ ]:
def read_config(config_path, section=None, param=None):
    """Read a nerfstudio config.yml file.
    
    Args:
        config_path: Path to config.yml
        section: If 'model', shows all model params. If None, shows everything.
        param: Specific model param name (e.g. 'seathru_from_iter'). Only used when section='model'.
    """
    with open(config_path, 'r') as f:
        config = yaml.unsafe_load(f)
    
    if section == 'model':
        model_config = config.pipeline.model
        # Convert the dataclass to a dict
        model_dict = vars(model_config)
        
        if param:
            if param in model_dict:
                print(f"{param}: {model_dict[param]}")
                return model_dict[param]
            else:
                print(f"Param '{param}' not found. Available params:")
                print(sorted(model_dict.keys()))
                return None
        else:
            for k, v in sorted(model_dict.items()):
                print(f"{k}: {v}")
            return model_dict
    
    elif section == 'optimizers':
        for k, v in config.optimizers.items():
            print(f"{k}: {v}")
        return config.optimizers
    
    else:
        print(vars(config))
        return config


In [ ]:
config_path = "/home/islabella/workspaces/irwin_ws/fyp-playground/outputs/torpedo_unprocessed/torpedo_unprocessed-baseline/sea-splatfacto/2026-02-27_045451/config.yml"

# All model params
read_config(config_path, section='model')

# One specific param
# read_config(config_path, section='model', param='seathru_from_iter')

# Everything
# read_config(config_path)

In [ ]:
def diff_configs(config_path_a, config_path_b, section=None, name_a="A", name_b="B"):
    """Diff two nerfstudio config.yml files.
    
    Args:
        config_path_a: Path to first config.yml
        config_path_b: Path to second config.yml
        section: 'model', 'optimizers', or None (top-level).
                 Defaults to 'model' since that's most useful.
        name_a: Label for first config
        name_b: Label for second config
    """
    with open(config_path_a, 'r') as f:
        config_a = yaml.unsafe_load(f)
    with open(config_path_b, 'r') as f:
        config_b = yaml.unsafe_load(f)
    
    def extract(config, section):
        if section == 'model':
            return vars(config.pipeline.model)
        elif section == 'optimizers':
            # Flatten optimizer dicts for comparison
            out = {}
            for group, opts in config.optimizers.items():
                opt = opts.get('optimizer')
                sch = opts.get('scheduler')
                if opt:
                    for k, v in vars(opt).items():
                        if k != '_target':
                            out[f"{group}.optimizer.{k}"] = v
                if sch:
                    for k, v in vars(sch).items():
                        if k != '_target':
                            out[f"{group}.scheduler.{k}"] = v
            return out
        else:
            return vars(config)
    
    dict_a = extract(config_a, section)
    dict_b = extract(config_b, section)
    
    all_keys = sorted(set(list(dict_a.keys()) + list(dict_b.keys())))
    
    changed, only_a, only_b = [], [], []
    
    for k in all_keys:
        in_a = k in dict_a
        in_b = k in dict_b
        
        if in_a and in_b:
            va, vb = dict_a[k], dict_b[k]
            # Compare as strings to handle complex objects
            if str(va) != str(vb):
                changed.append((k, va, vb))
        elif in_a:
            only_a.append((k, dict_a[k]))
        else:
            only_b.append((k, dict_b[k]))
    
    section_label = section or "top-level"
    
    if not changed and not only_a and not only_b:
        print(f"[{section_label}] Configs are identical.")
        return
    
    if changed:
        print(f"=== Changed ({section_label}) ===")
        max_key_len = max(len(k) for k, _, _ in changed)
        for k, va, vb in changed:
            print(f"  {k:<{max_key_len}}  {name_a}: {va}")
            print(f"  {'':<{max_key_len}}  {name_b}: {vb}")
            print()
    
    if only_a:
        print(f"=== Only in {name_a} ({section_label}) ===")
        for k, v in only_a:
            print(f"  {k}: {v}")
        print()
    
    if only_b:
        print(f"=== Only in {name_b} ({section_label}) ===")
        for k, v in only_b:
            print(f"  {k}: {v}")
    
    return {"changed": changed, "only_a": only_a, "only_b": only_b}

In [ ]:
path_a = "/home/islabella/workspaces/irwin_ws/fyp-playground/outputs/saltpond_unprocessed/saltpond_unprocessed-baseline/sea-splatfacto/2026-02-19_143146/config.yml"
path_b = "/home/islabella/workspaces/irwin_ws/fyp-playground/outputs/saltpond_unprocessed/saltpond_unprocessed-sweep_gw-loss-lambda/sea-splatfacto/2026-02-22_053351/config.yml"

# diff_configs(path_a, path_b, section=None, name_a="a", name_b="b")

# Diff model params (most common use case)
diff_configs(path_a, path_b, section='model', name_a="a", name_b="b")

# Diff optimizers
diff_configs(path_a, path_b, section='optimizers', name_a="a", name_b="b")